# MSRE Verification: Warehouse Case Study

An **industry-scale warehouse outbound process model** with 6 stations: PickA, PickB (parallel picking zones), Label, Scan, Pack, Release (serial processing). Based on operational data from a third-party logistics company. Stations have heterogeneous parameters (service times from 1.0 to 288.8 minutes, up to 4 workers per station).

## Warehouse Process Overview

The warehouse has 6 service stations:
- **S1**: Pick Zone A (4 servers)
- **S2**: Pick Zone B (2 servers)
- **S3**: Label (3 servers)
- **S4**: Scan (1 server)
- **S5**: Pack (3 servers)
- **S6**: Release (2 servers)

Jobs enter through 4 creation streams (2 for Zone A, 2 for Zone B), then flow through Label, Scan, Pack, Release.

## Macro-Step Refinement Equivalence (MSRE)

Each verification cell below confirms that two formalism translations produce **identical observable labels at every tick boundary**.

**Observation level W** = {queue_i_nonempty, server_i_busy} for i in {1, ..., 6}.

**Notebook parameters**: `seed=42`, `run_length=100.0`.

In [ ]:
import json
import simasm
assert simasm.__version__ >= "0.8.0", f"Requires simasm >= 0.8.0, got {simasm.__version__}"

## 1. DES Model Specifications

The warehouse outbound process is specified in three formalisms, each as a JSON document following SimASM's converter schema. The JSON files are stored under `input/warehouse/`.

| Formalism | JSON key concept | File | Lines |
|-----------|-----------------|------|-------|
| Event Graph (Schruben 1983) | `vertices` + `scheduling_edges` | `warehouse_eg.json` | 781 |
| ACD (Tocher 1960) | `activities` + `queues` | `warehouse_acd.json` | 649 |
| DEVS (Zeigler et al. 2019) | `atomic_models` + `coupled_model` | `warehouse_devs.json` | 2787 |

### 1a. Event Graph JSON

In [ ]:
INPUT_DIR = "../input/warehouse"

with open(f"{INPUT_DIR}/warehouse_eg.json") as f:
    warehouse_eg_json = json.load(f)

print(f"Model: {warehouse_eg_json['model_name']}")
print(f"Vertices: {len(warehouse_eg_json['vertices'])}")
print(f"Scheduling edges: {len(warehouse_eg_json['scheduling_edges'])}")
print(f"State variables: {list(warehouse_eg_json['state_variables'].keys())}")
print(f"Random streams: {list(warehouse_eg_json['random_streams'].keys())}")

### 1b. ACD JSON

In [ ]:
with open(f"{INPUT_DIR}/warehouse_acd.json") as f:
    warehouse_acd_json = json.load(f)

print(f"Model: {warehouse_acd_json['model_name']}")
print(f"Activities: {len(warehouse_acd_json['activities'])}")
print(f"Queues: {list(warehouse_acd_json['queues'].keys())}")
print(f"Random streams: {list(warehouse_acd_json['random_streams'].keys())}")

### 1c. DEVS JSON

In [ ]:
with open(f"{INPUT_DIR}/warehouse_devs.json") as f:
    warehouse_devs_json = json.load(f)

print(f"Model: {warehouse_devs_json['model_name']}")
print(f"Atomic models: {len(warehouse_devs_json['atomic_models'])}")
print(f"Coupled model couplings: {len(warehouse_devs_json['coupled_model']['couplings'])}")
print(f"Parameters: {len(warehouse_devs_json['parameters'])}")

## 2. Translation: JSON to SimASM

Each JSON specification is translated to SimASM code by a formalism-specific converter. The converter maps the formalism's execution algorithm to ASM transition rules:

- **EG converter** (`simasm.converter.event_graph`): implements the Next-Event Time-Advance algorithm. Each vertex becomes an event rule; scheduling edges become conditional FEL insertions.
- **ACD converter** (`simasm.converter.acd`): implements the Three-Phase Activity Scanning algorithm. Each activity becomes a scan-and-execute rule pair; queues map to marking functions.
- **DEVS converter** (`simasm.converter.devs`): implements the Abstract Simulator algorithm. Each atomic model yields internal/external transition rules; the coordinator handles output routing and time advance.

Below we run the converters and register the resulting models for verification.

In [ ]:
%%simasm convert

convert warehouse_eg:
    source: "../input/warehouse/warehouse_eg.json"
    formalism: event_graph
    register: "warehouse_eg"
    print: 30
endconvert

In [ ]:
%%simasm convert

convert warehouse_acd:
    source: "../input/warehouse/warehouse_acd.json"
    formalism: acd
    register: "warehouse_acd"
    print: 30
endconvert

In [ ]:
%%simasm convert

convert warehouse_devs:
    source: "../input/warehouse/warehouse_devs.json"
    formalism: devs
    register: "warehouse_devs"
    print: 30
endconvert

## 3. MSRE Verification

With all three models converted and registered, we verify macro-step refinement equivalence (MSRE) across all three formalism pairs. Each verify cell imports the registered model by name.

In [ ]:
%%simasm verify

verification warehouse_eg_acd_msre:

    models:
        import EG from "warehouse_eg"
        import ACD from "warehouse_acd"
    endmodels

    seed: 42

    labels:
        label queue_1_nonempty for EG: "Q1 > 0"
        label queue_1_nonempty for ACD: "marking(Q1) > 0"
        label server_1_busy for EG: "S1 < S1_capacity"
        label server_1_busy for ACD: "marking(S1) < S1_capacity"
        label queue_2_nonempty for EG: "Q2 > 0"
        label queue_2_nonempty for ACD: "marking(Q2) > 0"
        label server_2_busy for EG: "S2 < S2_capacity"
        label server_2_busy for ACD: "marking(S2) < S2_capacity"
        label queue_3_nonempty for EG: "Q3 > 0"
        label queue_3_nonempty for ACD: "marking(Q3) > 0"
        label server_3_busy for EG: "S3 < S3_capacity"
        label server_3_busy for ACD: "marking(S3) < S3_capacity"
        label queue_4_nonempty for EG: "Q4 > 0"
        label queue_4_nonempty for ACD: "marking(Q4) > 0"
        label server_4_busy for EG: "S4 < S4_capacity"
        label server_4_busy for ACD: "marking(S4) < S4_capacity"
        label queue_5_nonempty for EG: "Q5 > 0"
        label queue_5_nonempty for ACD: "marking(Q5) > 0"
        label server_5_busy for EG: "S5 < S5_capacity"
        label server_5_busy for ACD: "marking(S5) < S5_capacity"
        label queue_6_nonempty for EG: "Q6 > 0"
        label queue_6_nonempty for ACD: "marking(Q6) > 0"
        label server_6_busy for EG: "S6 < S6_capacity"
        label server_6_busy for ACD: "marking(S6) < S6_capacity"
    endlabels

    observables:
        observable queue_1_nonempty:
            EG -> queue_1_nonempty
            ACD -> queue_1_nonempty
        endobservable
        observable server_1_busy:
            EG -> server_1_busy
            ACD -> server_1_busy
        endobservable
        observable queue_2_nonempty:
            EG -> queue_2_nonempty
            ACD -> queue_2_nonempty
        endobservable
        observable server_2_busy:
            EG -> server_2_busy
            ACD -> server_2_busy
        endobservable
        observable queue_3_nonempty:
            EG -> queue_3_nonempty
            ACD -> queue_3_nonempty
        endobservable
        observable server_3_busy:
            EG -> server_3_busy
            ACD -> server_3_busy
        endobservable
        observable queue_4_nonempty:
            EG -> queue_4_nonempty
            ACD -> queue_4_nonempty
        endobservable
        observable server_4_busy:
            EG -> server_4_busy
            ACD -> server_4_busy
        endobservable
        observable queue_5_nonempty:
            EG -> queue_5_nonempty
            ACD -> queue_5_nonempty
        endobservable
        observable server_5_busy:
            EG -> server_5_busy
            ACD -> server_5_busy
        endobservable
        observable queue_6_nonempty:
            EG -> queue_6_nonempty
            ACD -> queue_6_nonempty
        endobservable
        observable server_6_busy:
            EG -> server_6_busy
            ACD -> server_6_busy
        endobservable
    endobservables

    check:
        type: macro_step_refinement
        run_length: 100.0
        timeout: 60
    endcheck

    output:
        format: "json"
        include_counterexample: true
    endoutput

endverification

In [ ]:
%%simasm verify

verification warehouse_eg_devs_msre:

    models:
        import EG from "warehouse_eg"
        import DEVS from "warehouse_devs"
    endmodels

    seed: 42

    labels:
        label queue_1_nonempty for EG: "Q1 > 0"
        label queue_1_nonempty for DEVS: "PickA_queue_count > 0"
        label server_1_busy for EG: "S1 < S1_capacity"
        label server_1_busy for DEVS: "PickA_server_count > 0"
        label queue_2_nonempty for EG: "Q2 > 0"
        label queue_2_nonempty for DEVS: "PickB_queue_count > 0"
        label server_2_busy for EG: "S2 < S2_capacity"
        label server_2_busy for DEVS: "PickB_server_count > 0"
        label queue_3_nonempty for EG: "Q3 > 0"
        label queue_3_nonempty for DEVS: "Label_queue_count > 0"
        label server_3_busy for EG: "S3 < S3_capacity"
        label server_3_busy for DEVS: "Label_server_count > 0"
        label queue_4_nonempty for EG: "Q4 > 0"
        label queue_4_nonempty for DEVS: "Scan_queue_count > 0"
        label server_4_busy for EG: "S4 < S4_capacity"
        label server_4_busy for DEVS: "Scan_server_count > 0"
        label queue_5_nonempty for EG: "Q5 > 0"
        label queue_5_nonempty for DEVS: "Pack_queue_count > 0"
        label server_5_busy for EG: "S5 < S5_capacity"
        label server_5_busy for DEVS: "Pack_server_count > 0"
        label queue_6_nonempty for EG: "Q6 > 0"
        label queue_6_nonempty for DEVS: "Release_queue_count > 0"
        label server_6_busy for EG: "S6 < S6_capacity"
        label server_6_busy for DEVS: "Release_server_count > 0"
    endlabels

    observables:
        observable queue_1_nonempty:
            EG -> queue_1_nonempty
            DEVS -> queue_1_nonempty
        endobservable
        observable server_1_busy:
            EG -> server_1_busy
            DEVS -> server_1_busy
        endobservable
        observable queue_2_nonempty:
            EG -> queue_2_nonempty
            DEVS -> queue_2_nonempty
        endobservable
        observable server_2_busy:
            EG -> server_2_busy
            DEVS -> server_2_busy
        endobservable
        observable queue_3_nonempty:
            EG -> queue_3_nonempty
            DEVS -> queue_3_nonempty
        endobservable
        observable server_3_busy:
            EG -> server_3_busy
            DEVS -> server_3_busy
        endobservable
        observable queue_4_nonempty:
            EG -> queue_4_nonempty
            DEVS -> queue_4_nonempty
        endobservable
        observable server_4_busy:
            EG -> server_4_busy
            DEVS -> server_4_busy
        endobservable
        observable queue_5_nonempty:
            EG -> queue_5_nonempty
            DEVS -> queue_5_nonempty
        endobservable
        observable server_5_busy:
            EG -> server_5_busy
            DEVS -> server_5_busy
        endobservable
        observable queue_6_nonempty:
            EG -> queue_6_nonempty
            DEVS -> queue_6_nonempty
        endobservable
        observable server_6_busy:
            EG -> server_6_busy
            DEVS -> server_6_busy
        endobservable
    endobservables

    check:
        type: macro_step_refinement
        run_length: 100.0
        timeout: 60
    endcheck

    output:
        format: "json"
        include_counterexample: true
    endoutput

endverification

In [ ]:
%%simasm verify

verification warehouse_acd_devs_msre:

    models:
        import ACD from "warehouse_acd"
        import DEVS from "warehouse_devs"
    endmodels

    seed: 42

    labels:
        label queue_1_nonempty for ACD: "marking(Q1) > 0"
        label queue_1_nonempty for DEVS: "PickA_queue_count > 0"
        label server_1_busy for ACD: "marking(S1) < S1_capacity"
        label server_1_busy for DEVS: "PickA_server_count > 0"
        label queue_2_nonempty for ACD: "marking(Q2) > 0"
        label queue_2_nonempty for DEVS: "PickB_queue_count > 0"
        label server_2_busy for ACD: "marking(S2) < S2_capacity"
        label server_2_busy for DEVS: "PickB_server_count > 0"
        label queue_3_nonempty for ACD: "marking(Q3) > 0"
        label queue_3_nonempty for DEVS: "Label_queue_count > 0"
        label server_3_busy for ACD: "marking(S3) < S3_capacity"
        label server_3_busy for DEVS: "Label_server_count > 0"
        label queue_4_nonempty for ACD: "marking(Q4) > 0"
        label queue_4_nonempty for DEVS: "Scan_queue_count > 0"
        label server_4_busy for ACD: "marking(S4) < S4_capacity"
        label server_4_busy for DEVS: "Scan_server_count > 0"
        label queue_5_nonempty for ACD: "marking(Q5) > 0"
        label queue_5_nonempty for DEVS: "Pack_queue_count > 0"
        label server_5_busy for ACD: "marking(S5) < S5_capacity"
        label server_5_busy for DEVS: "Pack_server_count > 0"
        label queue_6_nonempty for ACD: "marking(Q6) > 0"
        label queue_6_nonempty for DEVS: "Release_queue_count > 0"
        label server_6_busy for ACD: "marking(S6) < S6_capacity"
        label server_6_busy for DEVS: "Release_server_count > 0"
    endlabels

    observables:
        observable queue_1_nonempty:
            ACD -> queue_1_nonempty
            DEVS -> queue_1_nonempty
        endobservable
        observable server_1_busy:
            ACD -> server_1_busy
            DEVS -> server_1_busy
        endobservable
        observable queue_2_nonempty:
            ACD -> queue_2_nonempty
            DEVS -> queue_2_nonempty
        endobservable
        observable server_2_busy:
            ACD -> server_2_busy
            DEVS -> server_2_busy
        endobservable
        observable queue_3_nonempty:
            ACD -> queue_3_nonempty
            DEVS -> queue_3_nonempty
        endobservable
        observable server_3_busy:
            ACD -> server_3_busy
            DEVS -> server_3_busy
        endobservable
        observable queue_4_nonempty:
            ACD -> queue_4_nonempty
            DEVS -> queue_4_nonempty
        endobservable
        observable server_4_busy:
            ACD -> server_4_busy
            DEVS -> server_4_busy
        endobservable
        observable queue_5_nonempty:
            ACD -> queue_5_nonempty
            DEVS -> queue_5_nonempty
        endobservable
        observable server_5_busy:
            ACD -> server_5_busy
            DEVS -> server_5_busy
        endobservable
        observable queue_6_nonempty:
            ACD -> queue_6_nonempty
            DEVS -> queue_6_nonempty
        endobservable
        observable server_6_busy:
            ACD -> server_6_busy
            DEVS -> server_6_busy
        endobservable
    endobservables

    check:
        type: macro_step_refinement
        run_length: 100.0
        timeout: 60
    endcheck

    output:
        format: "json"
        include_counterexample: true
    endoutput

endverification

## Summary

All verification cells above should return **EQUIVALENT**. This confirms macro-step refinement equivalence across all three formalism pairs for the warehouse case study.

The notebook demonstrated the full pipeline:
1. **Specification**: Three DES formalisms (EG, ACD, DEVS) expressed as JSON
2. **Translation**: Mechanical conversion to SimASM via `%%simasm convert`
3. **Verification**: Pairwise MSRE checking via `%%simasm verify`